# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MitudruDutta/FlyRankAI/blob/main/Week%203/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook establishes the formal **Data Contract** for our lane (**Refresh / Content Opportunity Scoring**):
1. **The Contract in Plain Words (5 core answers):** Unit of analysis, tables, observation windows, prediction target, and deliberate exclusions.
2. **Field Categorization:** Rigidly classifying every column as Feature, Label, Context, or Excluded.
3. **Verification Queries (DuckDB SQL):** Empirical proof of Grain, Row Counts & Date Spans, Availability filtered with `IS TRUE`, our 5 core features, and the deliberate Leakage Trap demonstration.
4. **Data Limitations:** Naming and quantifying structural blind spots (unbalanced client history and GA4 sparsity).
5. **Self-Check:** Honest verification of all contract deliverables.

> Loaded skills: `skills/writing-data-contracts/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md`.

## 1. Unit of analysis + time window

### The Five Contract Answers in Plain Words

1. **What one row means for this lane:**
   Exactly **one unique published content item (`content_id`) belonging to a client domain (`client_id`) evaluated over a fixed 30-day baseline decision window**.
2. **Which table(s) we will use:**
   - In the hosted warehouse: `fact_content_daily_performance` (partitioned by month, selecting mid-panel month `month=2026-03` to avoid peeking into the final test month), joined with `dim_content` (metadata) and `dim_clients` (client coverage).
   - In our validated reference playground: `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages across 32 clients).
3. **Which time window:**
   - **Baseline Feature Window:** 30-day historical observation window (`2026-03-01` to `2026-03-31`) with prior comparison period (`2026-02-01` to `2026-02-28`).
   - **Target / Outcome Window:** Subsequent 30-day evaluation window (`2026-04-01` to `2026-04-30`) to observe forward performance trajectory.
4. **What we predict or rank (label or proxy):**
   - **Target:** `is_declining_label` — an observed empirical drop of $>20\%$ in search impressions in the forward window (`trend_direction == 'down'`).
   - **Output:** A ranked review queue sorted by opportunity score, evaluated on **Precision@50**.
5. **One thing we deliberately EXCLUDE (and why):**
   - **Future outcome-derived columns:** `trend_direction` and `trend_pct` (as well as forward-window metrics `impressions_last_30d`).
   - **Rationale:** Because `trend_direction` is directly computed from `trend_pct`, including either as an input feature constitutes catastrophic **target leakage**—the model simply memorizes the answer in disguise rather than learning predictive signals.

In [1]:
import os, sys
import duckdb
import pandas as pd
import numpy as np

# Resolve dataset path across directory structures
candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../Week 1/data/raw/content_refresh_anonymized.csv",
    "Week 1/data/raw/content_refresh_anonymized.csv",
    "../Week 1/data/raw/content_refresh_anonymized.csv",
    os.path.expanduser("~/Documents/FlyRankAI/Week 1/data/raw/content_refresh_anonymized.csv")
]
DATA_PATH = next((p for p in candidates if os.path.exists(p)), None)
assert DATA_PATH is not None, "Starter dataset CSV not found in search paths."

con = duckdb.connect()
df_raw = pd.read_csv(DATA_PATH)
con.register("dataset", df_raw)

print(f"Data source resolved: {DATA_PATH}")
print(f"Registered table 'dataset' with {len(df_raw):,} rows and {len(df_raw.columns)} columns in DuckDB.")


Data source resolved: /home/btwitsvoid/Documents/FlyRankAI/Week 1/data/raw/content_refresh_anonymized.csv
Registered table 'dataset' with 30,000 rows and 44 columns in DuckDB.


## 2. Fields: feature / label / context / excluded

Every column in our data contract is classified into exactly one of four strict operational categories:

| Category | Columns | Operational Rule & Definition |
|---|---|---|
| **Features** | `impressions_90d`, `avg_position`, `ctr`, `days_since_last_update`, `content_age_days`, `word_count`, `engagement_rate` | Observable facts knowable *before* the decision moment. Safe for model training. |
| **Label / Proxy** | `is_declining_label` | Derived from forward search trajectory: `(trend_direction == 'down').astype(int)`. Ground truth to predict; never an input. |
| **Context** | `content_id`, `client_id`, `content_type`, `position_tier`, `main_intent` | Metadata used strictly for joins, grouping, client-holdout validation splits, and audit slices. Never learned directly. |
| **Excluded** | `trend_pct`, `trend_direction`, `impressions_last_30d`, `clicks_last_30d`, `provider_used`, `model_used` | **Target leakage** (`trend_pct`, `trend_direction` contain the target formula) or **metadata with extreme missingness / non-search bias** (`provider_used`, `model_used`). |

In [2]:
# Enforce contract field separation in code
feature_cols = ["impressions_90d", "avg_position", "ctr", "days_since_last_update", "content_age_days"]
label_cols = ["is_declining_label"]
context_cols = ["content_id", "client_id", "content_type", "position_tier"]
excluded_cols = ["trend_pct", "trend_direction", "impressions_last_30d", "clicks_last_30d", "provider_used", "model_used"]

# Audit check: Ensure zero overlap between features and excluded/label sets
assert len(set(feature_cols).intersection(set(excluded_cols))) == 0, "CONTRACT BREACH: Excluded column found in features!"
assert len(set(feature_cols).intersection(set(label_cols))) == 0, "CONTRACT BREACH: Label column found in features!"
print(f"Contract Field Audit Passed: {len(feature_cols)} features, {len(label_cols)} label, {len(context_cols)} context, {len(excluded_cols)} excluded.")


Contract Field Audit Passed: 5 features, 1 label, 4 context, 6 excluded.


## 3. Verify it with queries (grain, counts, missing values, windows)

We verify all contract claims empirically using DuckDB SQL:
1. **Query 1 (The Grain Check):** Prove that `content_id` is unique with zero duplicate records (`HAVING COUNT(*) > 1` returns 0 rows).
2. **Query 2 (Slice Row Count and Date Span):** Prove that our slice contains exactly 30,000 rows across 32 clients, and span from 90 to 564 days of content age.
3. **Query 3 (Availability with `IS TRUE`):** Filter with explicit `IS TRUE` to inspect active GSC exposure and GA4 analytics presence.
4. **Five Core Features:** Extract a 5-feature dataframe with an explicit "knowable when?" line for each.
5. **The Leakage Trap:** Deliberately inject `trend_pct`, demonstrate an artificial jump to 1.000 Precision@50, then purge it to preserve the honest 0.720 baseline.

In [3]:
# --- Query 1: Prove the Grain (Zero duplicate keys) ---
q1_grain = con.sql("""
SELECT 
    content_id, 
    COUNT(*) as duplicate_count
FROM dataset
GROUP BY content_id
HAVING duplicate_count > 1
LIMIT 5;
""").df()

print("--- Query 1: Grain Verification (Duplicate keys found) ---")
print(f"Duplicate rows returned: {len(q1_grain)}")
assert len(q1_grain) == 0, "Grain violation: content_id is not unique!"
print("Verified: Grain is strictly exactly 1 row per content_id.\n")

# --- Query 2: Slice Row Count & Date/Age Span ---
q2_counts = con.sql("""
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT client_id) as n_clients,
    MIN(content_age_days) as min_age_days,
    MAX(content_age_days) as max_age_days,
    ROUND(AVG(content_age_days), 1) as avg_age_days
FROM dataset;
""").df()

print("--- Query 2: Row Count & Age Span ---")
print(q2_counts.to_string(index=False))
print("\n")

# --- Query 3: Availability (Filtered with IS TRUE) ---
q3_availability = con.sql("""
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN (impressions_90d > 0) IS TRUE THEN 1 ELSE 0 END) as gsc_active_rows,
    ROUND(SUM(CASE WHEN (impressions_90d > 0) IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as gsc_active_pct,
    SUM(CASE WHEN (engaged_sessions_90d > 0) IS TRUE THEN 1 ELSE 0 END) as ga4_active_rows,
    ROUND(SUM(CASE WHEN (engaged_sessions_90d > 0) IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as ga4_active_pct
FROM dataset;
""").df()

print("--- Query 3: Availability via IS TRUE ---")
print(q3_availability.to_string(index=False))
print("Observed: GSC search visibility is 100% available, while GA4 engagement is active on 27.9% of rows.\n")

# --- Five Features Frame with 'Knowable When?' Rationale ---
# Feature 1: impressions_90d    -> Knowable at decision moment because GSC logs search impressions prior to monthly review.
# Feature 2: avg_position        -> Knowable at decision moment because rank telemetry is logged daily up to the cutoff.
# Feature 3: ctr                 -> Knowable at decision moment because historical clicks/impressions are finalized facts.
# Feature 4: days_since_update   -> Knowable at decision moment because CMS modification timestamp is directly queryable.
# Feature 5: content_age_days    -> Knowable at decision moment because article creation metadata is permanently stored.

five_features_df = con.sql("""
SELECT 
    content_id,
    impressions_90d,
    avg_position,
    ctr,
    days_since_last_update,
    content_age_days
FROM dataset
LIMIT 5;
""").df()

print("--- Five Core Features Frame ---")
print(five_features_df.to_string(index=False))
print("\n")

# --- Four: The Trap (Deliberate Leakage Experiment) ---
from sklearn.tree import DecisionTreeClassifier

df_leak = df_raw.copy()
y = df_leak["trend_direction"].str.lower().eq("down").astype(int).values

def calc_p50(scores, labels):
    idx = np.argsort(-np.asarray(scores))[:50]
    return float(np.asarray(labels)[idx].mean())

# 1. Honest Model
X_honest = df_leak[feature_cols].fillna(0).values
tree_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_p50 = calc_p50(tree_honest.predict_proba(X_honest)[:, 1], y)

# 2. Leaky Model (Springing the trap by injecting trend_pct)
X_leaky = df_leak[feature_cols + ["trend_pct"]].fillna(0).values
tree_leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_p50 = calc_p50(tree_leaky.predict_proba(X_leaky)[:, 1], y)

print("--- The Leakage Trap Demonstration ---")
print(f"Honest Feature Set Precision@50:       {honest_p50:.3f}")
print(f"Leaky Feature Set (+trend_pct) P@50:   {leaky_p50:.3f}  <-- Artificial 1.000 jump!")
print("Purging trend_pct from model features: Restored honest Precision@50 = " + f"{honest_p50:.3f}.")


--- Query 1: Grain Verification (Duplicate keys found) ---
Duplicate rows returned: 0
Verified: Grain is strictly exactly 1 row per content_id.

--- Query 2: Row Count & Age Span ---
 total_rows  n_clients  min_age_days  max_age_days  avg_age_days
      30000         32            90           564         256.2


--- Query 3: Availability via IS TRUE ---
 total_rows  gsc_active_rows  gsc_active_pct  ga4_active_rows  ga4_active_pct
      30000          30000.0           100.0           8371.0            27.9
Observed: GSC search visibility is 100% available, while GA4 engagement is active on 27.9% of rows.

--- Five Core Features Frame ---
          content_id  impressions_90d  avg_position  ctr  days_since_last_update  content_age_days
content_304f48230142             3803          10.6 0.76                      20               187
content_a1fb4e703a9e            15320          20.3 0.05                      25               445
content_9aa793d4d895            12581          36.5 0.09

--- The Leakage Trap Demonstration ---
Honest Feature Set Precision@50:       0.720
Leaky Feature Set (+trend_pct) P@50:   1.000  <-- Artificial 1.000 jump!
Purging trend_pct from model features: Restored honest Precision@50 = 0.720.


## 4. Data limits

### Named Structural Limitation of this Slice: **Unbalanced History & Analytics Sparsity**

1. **Unbalanced Tracking Setup Across Clients:**
   - In production data, clients onboard at different times. Some domains have full 17-month GSC history (`gsc_data_start`), while others have only 90 days.
   - Rows recorded prior to a client's analytics onboarding date (`ga4_data_start`) are zero-filled with `ga4_data_available = FALSE`.
2. **The Zero-Engagement Trap:**
   - As proven in Query 3, only **27.9% of rows** have active GA4 engaged sessions. A naive model that treats zeros as "zero user engagement" rather than "analytics not instrumented" will learn false negative signals.
   - Therefore, all analytics-based signals must be conditioned on `ga4_data_available IS TRUE` or accompanied by missingness indicator flags (`has_ga4_flag`).

In [4]:
# Verify client-level variance in tracking and volume
client_variation = con.sql("""
SELECT 
    client_id,
    COUNT(*) as total_pages,
    ROUND(AVG(impressions_90d), 1) as avg_impressions,
    ROUND(SUM(CASE WHEN engaged_sessions_90d > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as pct_pages_with_ga4
FROM dataset
GROUP BY client_id
ORDER BY total_pages DESC
LIMIT 6;
""").df()

print("--- Client-Level Tracking Disparities (Proof of Limitation) ---")
print(client_variation.to_string(index=False))
print("\nObservation: Notice that client engagement instrumentation varies wildly (from 50%+ down to <10%).")


--- Client-Level Tracking Disparities (Proof of Limitation) ---
        client_id  total_pages  avg_impressions  pct_pages_with_ga4
client_19581e27de         7008           8058.6                31.7
client_6208ef0f77         3681           9601.8                53.4
client_4e07408562         2294           8599.4                42.9
client_3fdba35f04         2267           3717.5                34.9
client_f369cb89fc         1796           2538.2                10.0
client_8527a891e2         1194            422.3                 6.2

Observation: Notice that client engagement instrumentation varies wildly (from 50%+ down to <10%).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.